# Resident Progress & Reintegration Readiness Pipeline

## 1. Executive Summary & Problem Framing
**Objective**: To develop a high-fidelity analytical pipeline that predicts resident readiness for reintegration and explains the underlying drivers of progress. This tool assists social workers in making evidence-based decisions for discharge and intervention planning.

**Key Questions**:
1. Which clinical and educational factors most strongly correlate with reintegration readiness?
2. Do improvement trends (e.g., rising health scores) predict readiness better than static snapshots?
3. Is there a diminishing return on the number of counseling sessions provided?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import statsmodels.api as sm
from datetime import datetime

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Data Preparation & Robust Merging
We integrate demographic, clinical (process recordings), educational, and health data to form a holistic view of each resident's journey.

In [ ]:
# Load datasets
residents = pd.read_csv('../lighthouse_csv_v7/residents.csv')
recordings = pd.read_csv('../lighthouse_csv_v7/process_recordings.csv')
education = pd.read_csv('../lighthouse_csv_v7/education_records.csv')
health = pd.read_csv('../lighthouse_csv_v7/health_wellbeing_records.csv')

# Convert dates to datetime objects
recordings['session_date'] = pd.to_datetime(recordings['session_date'])
education['record_date'] = pd.to_datetime(education['record_date'])
health['record_date'] = pd.to_datetime(health['record_date'])

print(f"Loaded {len(residents)} residents.")

## 3. Advanced Feature Engineering
### 3.1 Aggregating Clinical Sessions
We calculate the intensity and quality of social work interventions.

In [ ]:
session_metrics = recordings.groupby('resident_id').agg({
    'recording_id': 'count',
    'session_duration_minutes': 'mean',
    'progress_noted': lambda x: (x == True).mean() * 100
}).rename(columns={
    'recording_id': 'session_count',
    'session_duration_minutes': 'avg_session_duration',
    'progress_noted': 'progress_noted_pct'
}).reset_index()

session_metrics.head()

### 3.2 Calculating Improvement Trends
We look at the trajectory of health and education over the last two recorded months to see if 'momentum' predicts readiness.

In [ ]:
def calculate_trend(df, id_col, date_col, value_col):
    """Calculates the difference between the most recent and second-most recent record."""
    df_sorted = df.sort_values([id_col, date_col])
    trends = {}
    
    for rid, group in df_sorted.groupby(id_col):
        if len(group) >= 2:
            # Difference between last and second to last
            val_last = group.iloc[-1][value_col]
            val_prev = group.iloc[-2][value_col]
            trends[rid] = val_last - val_prev
        else:
            trends[rid] = 0 # No trend possible
            
    return pd.Series(trends, name=f"{value_col}_trend")

health_trend = calculate_trend(health, 'resident_id', 'record_date', 'general_health_score')
edu_trend = calculate_trend(education, 'resident_id', 'record_date', 'attendance_rate')

# Get the most recent values as static features as well
last_health = health.sort_values('record_date').groupby('resident_id')['general_health_score'].last().rename('current_health_score')
last_edu = education.sort_values('record_date').groupby('resident_id')['attendance_rate'].last().rename('current_attendance_rate')

# Merge engineered features
df = residents.merge(session_metrics, on='resident_id', how='left')
df = df.merge(health_trend, left_on='resident_id', right_index=True, how='left')
df = df.merge(edu_trend, left_on='resident_id', right_index=True, how='left')
df = df.merge(last_health, on='resident_id', how='left')
df = df.merge(last_edu, on='resident_id', how='left')

# Final Cleaning
df['is_ready'] = (df['reintegration_status'] == 'Completed').astype(int)
df['length_of_stay_days'] = (pd.to_datetime('today') - pd.to_datetime(df['date_of_admission'])).dt.days

# Fill metrics with 0 or mean
metric_cols = ['session_count', 'avg_session_duration', 'progress_noted_pct', 
               'general_health_score_trend', 'attendance_rate_trend', 
               'current_health_score', 'current_attendance_rate']
df[metric_cols] = df[metric_cols].fillna(0)

print("Feature engineering complete.")

## 4. Exploration & Visualization
Visualizing the complex interplay of health, education, and social work.

In [ ]:
# 4.1 Heatmap of clinical and educational correlations
plt.figure(figsize=(10, 8))
corr = df[['current_health_score', 'current_attendance_rate', 'session_count', 
           'progress_noted_pct', 'general_health_score_trend', 'is_ready']].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap: Holistic Progress Metrics')
plt.show()

# 4.2 Boxplot: Length of Stay by Readiness
plt.figure(figsize=(10, 6))
sns.boxplot(x='is_ready', y='length_of_stay_days', data=df)
plt.title('Tenure (Length of Stay) by Reintegration Readiness')
plt.xticks([0, 1], ['In Progress', 'Ready/Completed'])
plt.show()

# 4.3 Risk Level Distribution over Readiness
plt.figure(figsize=(10, 6))
sns.countplot(x='current_risk_level', hue='is_ready', data=df, 
              order=['Low', 'Medium', 'High', 'Critical'])
plt.title('Risk Level Distribution vs Readiness Status')
plt.legend(title='Is Ready', labels=['No', 'Yes'])
plt.show()

## 5. Modeling
### 5.1 Predictive Modeling: Gradient Boosting
We use Gradient Boosting to capture non-linear relationships and interactions between features.

In [ ]:
features = ['session_count', 'avg_session_duration', 'progress_noted_pct', 
            'general_health_score_trend', 'attendance_rate_trend', 
            'current_health_score', 'current_attendance_rate', 'length_of_stay_days']

X = df[features]
y = df['is_ready']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Feature Importance
plt.figure(figsize=(10, 6))
feat_importances = pd.Series(gb.feature_importances_, index=features).sort_values(ascending=True)
feat_importances.plot(kind='barh', color='teal')
plt.title('Predictive Feature Importance (Gradient Boosting)')
plt.show()

### 5.2 Explanatory Modeling: Logistic Regression
We use `statsmodels` to interpret the statistical significance of each driver.

In [ ]:
X_logit = sm.add_constant(X)
logit_model = sm.Logit(y, X_logit).fit(disp=0)
print(logit_model.summary())

# Visualizing Odds Ratios
params = logit_model.params[1:]
conf = logit_model.conf_int().iloc[1:]
conf.columns = ['OR_low', 'OR_high']
odds_ratios = pd.concat([params, conf], axis=1)
odds_ratios.columns = ['Logit_Coeff', 'Lower_CI', 'Upper_CI']
odds_ratios = np.exp(odds_ratios) # Convert to Odds Ratios

print("\n--- Interpretation: Session Count vs Readiness ---")
session_coeff = logit_model.params['session_count']
print(f"The coefficient for session_count is {session_coeff:.4f}.")
if session_coeff > 0:
    print("Each additional session increases the log-odds of readiness, but social work theory suggests that after a certain point (diminishing returns), the impact may plateau or even decrease if the resident becomes overly dependent or the case is stagnating.")

## 6. Causal & Case Category Analysis
Why do 'Trafficked' residents often have longer timelines than 'Neglected' residents?

1. **Trauma Complexity**: Trafficked individuals often experience deep psychological coercion and organized exploitation, requiring longer stabilization periods before vocational or educational progress can be sustained.
2. **Legal Complications**: Trafficking cases often involve ongoing legal proceedings, which can delay 'Completed' reintegration status as the resident remains in the care system as a witness or for protection.
3. **Safety Risks**: Reintegration for 'Trafficked' residents requires more rigorous environmental vetting to ensure they are not re-trafficked, leading to naturally longer 'In Progress' states compared to 'Neglected' cases where family reunification might be simpler.

## 7. Deployment & Actionable Recommendations

Based on the model findings and trends, we recommend the following for the social work team:

1. **Trend-Based Alerts**: If a resident's `general_health_score` drops by more than 0.5 in a single month (negative trend), trigger an automated 'Clinical Review' alert, regardless of their current static score.
2. **The 'Sweet Spot' for Sessions**: Monitor residents who exceed 2 standard deviations above the mean `session_count`. If `progress_noted_pct` is below 50% for these residents, consider rotating the assigned social worker or changing the intervention modality (e.g., from individual to group sessions) to overcome diminishing returns.
3. **Readiness Scoreboard**: Implement a dashboard showing 'Probability of Readiness' using the Gradient Boosting model. Residents with >75% probability should be prioritized for their final reintegration assessment to ensure efficient throughput and free up safehouse capacity.

In [ ]:
import joblib, os

os.makedirs('models', exist_ok=True)

# Export the trained Gradient Boosting model and feature columns
joblib.dump(gb, 'models/resident_progress_model.joblib')
joblib.dump(features, 'models/resident_progress_features.joblib')

print("Exported: models/resident_progress_model.joblib")
print(f"Feature count: {len(features)}")